# Applied Data Science Capstone — 3. EDA with SQL

## Objective
Load the launch records into a SQL table **`SPACEXTBL`** (SQLite) and answer
business questions with SQL — the same queries as the course's EDA-SQL lab.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/dataset_spacex.csv")
con = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", con, index=False, if_exists="replace")

def q(sql, title):
    print(f"--- {title}")
    print(sql.strip())
    out = pd.read_sql(sql, con)
    print(out.to_string(index=False))
    print()

q("SELECT COUNT(*) AS launches FROM SPACEXTBL", "Q0. Row count")

--- Q0. Row count
SELECT COUNT(*) AS launches FROM SPACEXTBL
 launches
       90



In [2]:
q("""SELECT DISTINCT LaunchSite AS unique_launch_sites
     FROM SPACEXTBL ORDER BY 1""",
  "Q1. Unique launch sites")

q("""SELECT LaunchSite, COUNT(*) AS launches
     FROM SPACEXTBL GROUP BY LaunchSite ORDER BY launches DESC""",
  "Q2. Launches per site")

q("""SELECT SUM(PayloadMass) AS total_payload_kg,
          ROUND(AVG(PayloadMass), 1) AS avg_payload_kg
     FROM SPACEXTBL""",
  "Q3. Total and average payload mass")

q("""SELECT BoosterVersion, ROUND(AVG(PayloadMass), 1) AS avg_payload_kg
     FROM SPACEXTBL GROUP BY BoosterVersion ORDER BY avg_payload_kg DESC""",
  "Q4. Average payload by booster version")

q("""SELECT MIN(Date) AS first_successful_landing_date
     FROM SPACEXTBL WHERE Class = 1""",
  "Q5. First successful landing (date)")

q("""SELECT Serial, COUNT(*) AS successful_landings
     FROM SPACEXTBL WHERE Class = 1
     GROUP BY Serial ORDER BY successful_landings DESC LIMIT 5""",
  "Q6. Most reused successful boosters")

q("""SELECT Orbit,
          COUNT(*) AS launches,
          SUM(Class) AS successful_landings,
          ROUND(100.0 * SUM(Class) / COUNT(*), 1) AS success_rate_pct
     FROM SPACEXTBL GROUP BY Orbit ORDER BY success_rate_pct DESC""",
  "Q7. Landing success rate by orbit")

q("""SELECT LaunchSite,
          COUNT(*) AS launches,
          ROUND(100.0 * SUM(Class) / COUNT(*), 1) AS success_rate_pct
     FROM SPACEXTBL GROUP BY LaunchSite ORDER BY success_rate_pct DESC""",
  "Q8. Landing success rate by launch site")

--- Q1. Unique launch sites
SELECT DISTINCT LaunchSite AS unique_launch_sites
     FROM SPACEXTBL ORDER BY 1
unique_launch_sites
        CCAFS LC-40
       CCAFS SLC-40
         KSC LC-39A
        VAFB SLC-4E

--- Q2. Launches per site
SELECT LaunchSite, COUNT(*) AS launches
     FROM SPACEXTBL GROUP BY LaunchSite ORDER BY launches DESC
  LaunchSite  launches
CCAFS SLC-40        30
 CCAFS LC-40        23
  KSC LC-39A        22
 VAFB SLC-4E        15

--- Q3. Total and average payload mass
SELECT SUM(PayloadMass) AS total_payload_kg,
          ROUND(AVG(PayloadMass), 1) AS avg_payload_kg
     FROM SPACEXTBL
 total_payload_kg  avg_payload_kg
         410475.4          4560.8

--- Q4. Average payload by booster version
SELECT BoosterVersion, ROUND(AVG(PayloadMass), 1) AS avg_payload_kg
     FROM SPACEXTBL GROUP BY BoosterVersion ORDER BY avg_payload_kg DESC
      BoosterVersion  avg_payload_kg
    Falcon 9 Block 5          5221.2
Falcon 9 Full Thrust          4697.7
       Falcon 9 v1.1  

### Findings
* The four launch sites and their Falcon 9 launch counts are confirmed by Q2.
* Q7 shows landing success varies strongly by orbit (heavier / higher-energy
  orbits such as GTO are harder to recover from).
* Q8 ranks the sites by first-stage recovery success — input to the site
  selection discussion in the Folium notebook.